# Export trusted checkpoint and verify ONNX parity

Attach Output from notebook 01 and Imagenette. Source code is embedded with a verified SHA256. At least 20 dev images are compared, including independent torchvision versus deployment preprocessing. No training runs here.

Use **Save Version → Save & Run All**. Download outputs from `/kaggle/working/`.


In [ ]:
from pathlib import Path
from io import BytesIO
import base64
import hashlib
import sys
import zipfile
import subprocess
SOURCE_BUNDLE_SHA256 = '38bf3f7894c08255547a8ac0d89aec61ecb149191de43e0f889fb91a4ac6e745'
SOURCE_BUNDLE_BASE64 = '...'
WORK = Path('/kaggle/working')
if not WORK.is_dir():
    raise RuntimeError('Run this notebook on Kaggle')
SOURCE_ROOT = WORK / 'miragehunter-source'
source_bytes = base64.b64decode(SOURCE_BUNDLE_BASE64, validate=True)
if hashlib.sha256(source_bytes).hexdigest() != SOURCE_BUNDLE_SHA256:
    raise RuntimeError('Embedded source bundle checksum mismatch')
with zipfile.ZipFile(BytesIO(source_bytes)) as bundle:
    for entry in bundle.infolist():
        target = (SOURCE_ROOT / entry.filename).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()):
            raise ValueError('Unsafe source archive path')
    bundle.extractall(SOURCE_ROOT)
sys.path[:0] = [str(SOURCE_ROOT / 'ai-service'), str(SOURCE_ROOT)]
requirements = SOURCE_ROOT / 'experiments/requirements-kaggle.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
print('Source:', SOURCE_ROOT)
print('Source bundle SHA256:', SOURCE_BUNDLE_SHA256)


## Configure
Edit the values below before running.

In [ ]:
from pathlib import Path
from experiments.scripts.export_onnx import export_and_verify
checkpoints = sorted(Path('/kaggle/input').rglob('mobilenetv3_imagenette_best.pth'))
if len(checkpoints) != 1:
    raise RuntimeError('Attach exactly one trusted training output, or set ARTIFACT_DIR explicitly')
ARTIFACT_DIR = checkpoints[0].parent
DATASET_ROOT = None


In [ ]:
ARTIFACTS = export_and_verify(ARTIFACT_DIR, dataset_root=DATASET_ROOT, samples=32)
print('Verified artifact bundle:', ARTIFACTS)


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/miragehunter-output', 'zip', str(ARTIFACTS))
